తరువాతి నోట్బుక్స్‌ను నడపడానికి, మీరు ఇంకా చేయలేదైతే, మీరు `text-embedding-ada-002` ను బేస్ మోడల్‌గా ఉపయోగించే ఒక మోడల్‌ను డిప్లాయ్ చేయాలి మరియు డిప్లాయ్‌మెంట్ పేరును .env ఫైల్లో `AZURE_OPENAI_EMBEDDINGS_ENDPOINT` గా సెట్ చేయాలి


In [ ]:
import os
import pandas as pd
import numpy as np
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

client = AzureOpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],  # this is also the default, it can be omitted
  api_version = "2024-10-21",
  azure_endpoint = os.environ['AZURE_OPENAI_ENDPOINT']
  )

model = os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']

SIMILARITIES_RESULTS_THRESHOLD = 0.75
DATASET_NAME = "../embedding_index_3m.json"


తదుపరి, మనం ఎంబెడ్డింగ్ సూచీని Pandas డేటాఫ్రేమ్‌లో లోడ్ చేయబోతున్నాం. ఎంబెడ్డింగ్ సూచీ `embedding_index_3m.json` అనే JSON ఫైల్‌లో నిల్వ చేయబడింది. ఎంబెడ్డింగ్ సూచీలో అక్టోబర్ 2023 చివరి వరకు ప్రతి YouTube ట్రాన్స్క్రిప్టుల కోసం ఎంబెడ్డింగ్లు ఉంటాయి.


In [ ]:
def load_dataset(source: str) -> pd.core.frame.DataFrame:
    # Load the video session index
    pd_vectors = pd.read_json(source)
    return pd_vectors.drop(columns=["text"], errors="ignore").fillna("")

తర్వాత, మేము `get_videos` అనే ఫంక్షన్‌ను సృష్టించబోతున్నాము, ఇది క్వెరి కోసం ఎంబెడింగ్ ఇండెక్స్‌లో శోధన చేస్తుంది. ఈ ఫంక్షన్ క్వెరిసাথে అత్యంత సమానమైన టాప్ 5 వీడియోలను దిగజార్చుతుంది. ఫంక్షన్ పనిచేసే విధానం ఇలా ఉంది:

1. ముందుగా, ఎంబెడింగ్ ఇండెక్స్ యొక్క ఒక నకలు సృష్టించబడుతుంది.
2. తరువాత, OpenAI ఎంబెడింగ్ API ఉపయోగించి క్వెరి కోసం ఎంబెడింగ్ లెక్కించబడుతుంది.
3. ఆ తర్వాత ఎంబెడింగ్ ఇండెక్స్‌లో `similarity` అనే కొత్త కాలమ్ సృష్టించబడుతుంది. `similarity` కాలమ్ క్వెరి ఎంబెడింగ్ మరియు ప్రతి వీడియో సెగ్మెంట్ ఎంబెడింగ్ మధ్య కోసైన్ సమానతను కలిగి ఉంటుంది.
4. తరువాత, `similarity` కాలమ్ ద్వారా ఎంబెడింగ్ ఇండెక్స్ ఫిల్టర్ చేయబడుతుంది. కోసైన్ సమానత 0.75 లేదా అంతకంటే ఎక్కువటి ఉన్న వీడియోలని మాత్రమే ఫిల్టర్ చేస్తుంది.
5. చివరిగా, `similarity` కాలమ్ ద్వారా ఎంబెడింగ్ ఇండెక్స్ ఆర్డర్ చేసి టాప్ 5 వీడియోలను తిరిగి ఇస్తుంది.


In [ ]:
def cosine_similarity(a, b):
    if len(a) > len(b):
        b = np.pad(b, (0, len(a) - len(b)), 'constant')
    elif len(b) > len(a):
        a = np.pad(a, (0, len(b) - len(a)), 'constant')
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def get_videos(
    query: str, dataset: pd.core.frame.DataFrame, rows: int
) -> pd.core.frame.DataFrame:
    # create a copy of the dataset
    video_vectors = dataset.copy()

    # get the embeddings for the query    
    query_embeddings = client.embeddings.create(input=query, model=model).data[0].embedding

    # create a new column with the calculated similarity for each row
    video_vectors["similarity"] = video_vectors["ada_v2"].apply(
        lambda x: cosine_similarity(np.array(query_embeddings), np.array(x))
    )

    # filter the videos by similarity
    mask = video_vectors["similarity"] >= SIMILARITIES_RESULTS_THRESHOLD
    video_vectors = video_vectors[mask].copy()

    # sort the videos by similarity
    video_vectors = video_vectors.sort_values(by="similarity", ascending=False).head(
        rows
    )

    # return the top rows
    return video_vectors.head(rows)

ఈ ఫంక్షన్ చాలా సులభం, ఇది కేవలం శోధన ప్రశ్న ఫలితాలను ప్రింట్ చేస్తుంది.


In [ ]:
def display_results(videos: pd.core.frame.DataFrame, query: str):
    def _gen_yt_url(video_id: str, seconds: int) -> str:
        """convert time in format 00:00:00 to seconds"""
        return f"https://youtu.be/{video_id}?t={seconds}"

    print(f"\nVideos similar to '{query}':")
    for _, row in videos.iterrows():
        youtube_url = _gen_yt_url(row["videoId"], row["seconds"])
        print(f" - {row['title']}")
        print(f"   Summary: {' '.join(row['summary'].split()[:15])}...")
        print(f"   YouTube: {youtube_url}")
        print(f"   Similarity: {row['similarity']}")
        print(f"   Speakers: {row['speaker']}")

1. మొదట, ఎంబెడింగ్ సూచికను Pandas డేటాఫ్రేమ్‌లో లోడ్ చేస్తారు.
2. తదుపరి, వినియోగదారుని ఒక ప్రశ్నను నమోదు చేయమని అడుగుతారు.
3. తరువాత `get_videos` ఫంక్షన్‌ను పిలిచి ఆ ప్రశ్నకు సంబంధించిన ఎంబెడింగ్ సూచికను శోధిస్తారు.
4. చివరిగా, ఫలితాలను చూపించడానికి `display_results` ఫంక్షన్‌ను పిలుస్తారు.
5. ఆ తర్వాత వినియోగదారుని మరో ప్రశ్నను నమోదు చేయమని అడుగుతారు. వినియోగదారు `exit` అని నమోదు చేసిన వరకు ఈ ప్రక్రియ కొనసాగుతుంది.

![](../../../../translated_images/te/notebook-search.1e320b9c7fcbb0bc.webp)

మీరు ఒక ప్రశ్నను నమోదు చేయమని అడగబడుతుంది. ఒక ప్రశ్నను నమోదు చేసి ఎంటర్ నొక్కండి. ఆప్లికేషన్ ఆ ప్రశ్నకు సంబంధించి ఉన్న వీడియోల జాబితాను సూచిస్తుంది. ఆప్లికేషన్ ప్రశ్నకు సమాధానం ఉన్న వీడియోలోని స్థలానికి లింకును కూడా అందిస్తుంది.

ఇక్కడ కొన్ని ప్రయత్నించాల్సిన ప్రశ్నలు ఉన్నాయి:

- Azure మెషీన్ లెర్నింగ్ అంటే ఏమిటి?
- కన్‌వల్యూసనల్ న్యూట్రల్ నెట్‌వర్క్స్ ఎలా పనిచేస్తాయి?
- న్యూట్రల్ నెట్‌వర్క్ అంటే ఏమిటి?
- Azure మెషీన్ లెర్నింగ్‌తో Jupyter నోట్బుక్స్‌ని ఉపయోగించవచ్చా?
- ONNX అంటే ఏమిటి?


In [ ]:
pd_vectors = load_dataset(DATASET_NAME)

# get user query from input
while True:
    query = input("Enter a query: ")
    if query == "exit":
        break
    videos = get_videos(query, pd_vectors, 5)
    display_results(videos, query)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
